In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import trackio
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn

keys = 'u', 'v', 'w', 'p_total'

def extract(real, keys):
    with h5py.File("/data1/dataToYouri/couette_dns_ml_f32.h5", 'r') as f:
        data = []
        for k in keys:
            data.append(np.array(f[real][k]))
        return np.stack(data, axis=1)

realizations = ['R13', 'R14', 'R15', 'R18', 'R19', 'R21', 'R22', 'R23', 'R26', 'R28', 'R3', 'R4', 'R5']
samplings = {real: np.s_[:-1:5000 // 300] if real in ["R3", "R4", "R5"] else np.s_[:] for real in realizations} 

raw_d = {real: extract(real, keys)[samplings[real]] for real in realizations}
all_data = np.concatenate(list(raw_d.values()), axis=0)
T, C, X, Y, Z = all_data.shape

# Test and val realizations choosen as they have roughly avg mean
val_realizations = ["R18", "R5"]
test_realizations = ["R3", "R14"]  # ONLY USE FOR FINAL BENCHMARK, ANY DECISION TAKEN AFTER PERFORMANCE IS ESTABLISHED ON THIS DS IS LEAKAGE
train_realizations = [real for real in realizations if real not in val_realizations + test_realizations]
val_npds = np.concatenate([raw_d[r] for r in val_realizations])
train_npds = np.concatenate([raw_d[r] for r in train_realizations])

device = "cuda"

class DS(Dataset):
    def __init__(self, d):
        self.x = torch.from_numpy(d[:, 3:, :, 0, :]).float().to(device)
        self.y = torch.from_numpy(d[:, :3, :, :, :]).float().to(device)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]

normalization_axes = (0, 2, 4,)
all_data_mean = all_data.mean(normalization_axes, keepdims=True)
all_data_std = all_data.std(normalization_axes, keepdims=True) + 1e-10

all_data_mean_gpu = torch.from_numpy(all_data_mean).to(device)
all_data_std_gpu = torch.from_numpy(all_data_std).to(device)

def normalize(data):
    return (data - all_data_mean) / all_data_std

def unnormalize_velocity(data):
    return (data * all_data_std[:, :3]) + all_data_mean[:, :3]

def unnormalize_velocity_gpu(data):
    return (data * all_data_std_gpu[:, :3]) + all_data_mean_gpu[:, :3]

val_ds = DS(normalize(val_npds))
train_ds = DS(normalize(train_npds))

def train(model, opti, loss, ds):
    model.train()
    total_loss = 0
    for x, y in ds:
        opti.zero_grad()
        pred = model(x)
        l = loss(y, pred)
        total_loss += l.item()
        l.backward()
        opti.step()
    return total_loss / len(ds)

def evaluate(model, loss, ds):
    model.eval()
    total_loss = 0
    total_unnormalized_loss = 0
    with torch.no_grad():
        for x, y in ds:
            pred = model(x)
            l = loss(pred, y)
            total_loss += l.item()
            total_unnormalized_loss += total_unnormalized_loss_gpu(pred, y).item()
    return total_loss / len(ds), total_unnormalized_loss / len(ds) 

def predict(model, flat_x_ds):
    model.eval()
    with torch.no_grad():
        pred = model(flat_x_ds)
    return pred

def total_unnormalized_loss_gpu(pred, target):
    unmzed_pred = unnormalize_velocity_gpu(pred)
    unmzed_target = unnormalize_velocity_gpu(target)
    return torch.nn.functional.mse_loss(unmzed_pred, unmzed_target, reduction="mean")

def explained_variance(preds, ys):
    unormalized_loss = total_unnormalized_loss_gpu(preds, ys)
    explained_var = 1.0 - (unormalized_loss / unmzd_y.var(unbiased=False))
    return explained_var


def train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device):
    
    opti = optim(model.parameters(), lr=lr)
    trackio.init(
        project=project_name,
        config={"epochs": epochs, "learning_rate": lr, "batch_size": batch_size}
    )
    src_test_var = np.var(val_npds)
    for epoch in range(epochs):
        train_error = train(model, opti, loss, train_dl)
        test_error, total_unnormalized_loss = evaluate(model, loss, val_dl)
        
        trackio.log({
            "epoch": epoch,
            "train_error": train_error,
            "test_error": test_error,
            "explained_variance": 1 - (total_unnormalized_loss / src_test_var),
        })
    
    trackio.finish()

batch_size = 4
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)



# train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device)

In [2]:
# define model

epochs = 20
loss = nn.MSELoss(reduction='mean')
lr = 1e-3
optim = torch.optim.Adam
project_name = "conv-2"

In [3]:


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding="same"),
            nn.SiLU(),
            nn.Conv3d(out_ch, out_ch, 3, padding="same"),
            nn.SiLU(),
        )
    def forward(self, x):
        return self.conv(x)


class M4(nn.Module):
    def __init__(self):
        super().__init__()
        self.pressure_encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding="same"),
            nn.SiLU(),
            nn.Conv2d(8, 32, 3, padding="same"),
            nn.SiLU(),
        )

        self.D = 33
        self.c3d = 32
        self.lift = nn.Conv2d(32, self.D * self.c3d, 3, padding="same")
        
        self.xd = 24
        self.zd = 24

        self.conv_block_1 = ConvBlock(self.c3d, self.c3d)
        self.conv_block_2 = ConvBlock(self.c3d, self.c3d)
        self.conv_block_3 = ConvBlock(self.c3d, self.c3d)
        self.output = nn.Conv3d(self.c3d, 3, 3, padding="same")
        

    def forward(self, pressure):
        x = self.pressure_encoder(pressure)
        x = self.lift(x)
        # reshape into Batch, channels, x, D, z, 
        x = x.reshape(-1, self.c3d, self.xd, self.D, self.zd)
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.conv_block_3(x)
        x = self.output(x)
        return x

model = M4().to(device)

In [4]:
x, y = next(iter(train_dl))
print(x.shape)

torch.Size([4, 1, 24, 24])


In [5]:
pred = model(x)

In [6]:
print(pred.shape == y.shape)
print(pred.shape)
print(y.shape)

True
torch.Size([4, 3, 24, 33, 24])
torch.Size([4, 3, 24, 33, 24])


In [7]:
train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device)

* Trackio project initialized: conv-2
* Trackio metrics logged to: /home/chancrin/.cache/huggingface/trackio
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: brave-forest-1


* Run finished. Uploading logs to Trackio (please wait...)


In [8]:
from ipywidgets import interact, IntSlider


normalized_preds = predict(model, val_dl.dataset.x)
cpu_preds = unnormalize_velocity_gpu(normalized_preds).cpu()
cpu_ys = unnormalize_velocity_gpu(val_dl.dataset.y).cpu()
cpu_pressure = val_dl.dataset.x.cpu()
diff = cpu_preds - cpu_ys
mean_diff_global = diff.mean((0, 1, 3))

@interact(
    sample_idx=IntSlider(min=0, max=len(val_dl.dataset.x), step=1, value=0, description="Sample idx"),
    y_idx=IntSlider(min=0, max=32, step=1, value=0, description="y idx"),
)
def plot_val_data(sample_idx, y_idx):
    preds = cpu_preds
    ys = cpu_ys
    pressure = cpu_pressure
    fig, ax = plt.subplots(4, 4)
    
    for comp in [0, 1, 2]:
        ax[0, comp].imshow(preds[sample_idx, comp, :, y_idx, :])
        ax[1, comp].imshow(ys[sample_idx, comp, :, y_idx, :])
        ax[2, comp].imshow(np.abs(diff[sample_idx, comp, :, y_idx, :]), cmap="magma")
        ax[3, comp].imshow(pressure[sample_idx, 0], cmap="cividis")
    for a in ax.flat:
        a.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax[0, 3].bar(["u", "v", "w"], np.abs(diff[sample_idx, :, :, y_idx, :]).mean((1, 2)))
    mean_diff_sample = diff[sample_idx].mean((1, 3))
    
    ax[1, 3].plot(mean_diff_sample[0])
    ax[1, 3].plot(mean_diff_sample[1])
    ax[1, 3].plot(mean_diff_sample[2])
    ax[2, 3].plot(mean_diff_global[0])
    ax[2, 3].plot(mean_diff_global[1])
    ax[2, 3].plot(mean_diff_global[2])
    plt.tight_layout()
    plt.show()
    


interactive(children=(IntSlider(value=0, description='Sample idx', max=600), IntSlider(value=0, description='y…